# Phase 5 — Clinical Text Branch
22. EyeCLIP text encoder on clinical text (disease_description.xlsx)
23. Extend fusion: image + metadata + text -> patient representation
24. Re-run few-shot + open-set with trimodal setup

**Confirmed data shape:** `disease_description.xlsx` has ONE row per class (8 rows total) -- every image belonging to a class gets the identical text embedding. This is a fixed per-class "diagnostic description" vector, not per-patient variation (unlike metadata, which varies per patient).

**Requires:** EyeCLIP's text encoder (same checkpoint/repo as Phase 0), `eyeclip_embeddings.npz` (Phase 1), fused image+metadata setup from Phase 3 (`fused_embeddings.npz`, or rebuild inline here), and the same split CSVs used throughout.

In [1]:
import pathlib
clip_path = pathlib.Path(r"E:\rop\codes\EyeCLIP\eyeclip\clip.py")  # your actual path
text = clip_path.read_text()
text = text.replace("from pkg_resources import packaging", "import packaging")
clip_path.write_text(text)

9800

In [3]:
import sys
sys.path.insert(0, r"E:\rop\codes\EyeCLIP")  # your actual path

import torch, eyeclip

In [4]:
import torch, eyeclip
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, roc_auc_score

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CKPT_PATH = r"E:\rop\codes\eyeclip.pt"           # CHANGE to actual path -- same checkpoint as Phase 0/1
DISEASE_DESC_PATH = r"E:\rop\dataset\ROP-VL\disease_description.xlsx"
EMB_PATH = r"E:\rop\codes\eyeclip_embeddings.zip"
FUSED_EMB_PATH = r"E:\rop\codes\fused_embeddings.npz"
SPLIT_DIR = Path(r"E:\rop\dataset\ROP-VL\split")  # CHANGE to actual path

model, preprocess = eyeclip.load("ViT-B/32", device=DEVICE, jit=False)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
model.load_state_dict(state, strict=False)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print("EyeCLIP loaded on", DEVICE)

e:\.venv\Lib\site-packages\torch\jit\_serialization.py:170: FutureWarning: `torch.jit.load` is not supported in Python 3.14+ and may break. Please switch to `torch.export`.
  warnings.warn(


EyeCLIP loaded on cpu


## Encode the 8 class descriptions with EyeCLIP's text encoder
**Explainability:** this produces exactly 8 text embedding vectors (one per class). Every image belonging to a class is assigned that class's single text embedding -- there is no per-image text variation to encode.

In [5]:
desc_df = pd.read_excel(DISEASE_DESC_PATH)
print(desc_df)

# EyeCLIP/CLIP-style tokenizers usually cap at 77 tokens -- check assumption, don't silently truncate
assert hasattr(eyeclip, "tokenize"), "eyeclip module has no tokenize() -- check the repo's actual text API"
tokens = eyeclip.tokenize(desc_df["description"].tolist(), truncate=True).to(DEVICE)

with torch.no_grad():
    text_embeddings = model.encode_text(tokens).float().cpu().numpy()  # [8, 512]

print("text_embeddings shape:", text_embeddings.shape)
class_to_text_emb = dict(zip(desc_df["disease"], text_embeddings))

             disease                                        description
0             normal  healthy; no findings; no lesion signs; no reti...
1        Stage 1 ROP  a thin white line at the vascular-avascular ju...
2        Stage 2 ROP  ridge with width and height that evolves from ...
3        Stage 3 ROP  extraretinal neovascular proliferation extends...
4        Stage 4 ROP  partial retinal detachment, either spares  or ...
5        Stage 5 ROP  total retinal detachment; open-funnel detachme...
6              A-ROP  severe vasoconstriction; capillary nonperfusio...
7  Laser-treated ROP  laser scar; Laser spots look like small white ...
text_embeddings shape: (8, 512)


## Load image embeddings, metadata-fused embeddings, and splits; attach per-row text embedding by class label

In [6]:
data = np.load(EMB_PATH)
X_train_img, X_val_img, X_test_img = data["X_train"], data["X_val"], data["X_test"]

fused = np.load(FUSED_EMB_PATH)
F_train, F_val, F_test = fused["F_train"], fused["F_val"], fused["F_test"]

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

for name, arr, df in [("X_train", X_train_img, train_df), ("X_val", X_val_img, val_df),
                       ("X_test", X_test_img, test_df), ("F_train", F_train, train_df),
                       ("F_val", F_val, val_df), ("F_test", F_test, test_df)]:
    assert len(arr) == len(df), f"{name} row count != matching split csv"

unknown_classes_check = set(train_df["categories"].unique()) - set(class_to_text_emb.keys())
assert not unknown_classes_check, f"classes with no text description: {unknown_classes_check}"

def attach_text_embeddings(df):
    """Row-order-preserving lookup: each row gets its class's fixed text embedding."""
    return np.stack([class_to_text_emb[c] for c in df["categories"].values])

T_train = attach_text_embeddings(train_df)
T_val   = attach_text_embeddings(val_df)
T_test  = attach_text_embeddings(test_df)
print("T_train shape:", T_train.shape)

T_train shape: (1411, 512)


## Trimodal fusion: image + metadata + text -> patient representation
Same pattern as Phase 3's fusion (concat -> linear -> ReLU -> dropout -> linear), extended to a three-way concat. Image embeddings stay frozen; only the fusion layers are trained.

**Important caveat to flag explicitly:** because text embedding is identical for every image within a class, it acts as a fixed "class anchor" the fusion layer can key on directly -- this can inflate apparent classification performance in a way that doesn't reflect real per-patient signal. Interpret trimodal gains with this in mind, and note it in the final report rather than presenting it as an unqualified win.

In [7]:
import torch.nn as nn

IMG_DIM = X_train_img.shape[1]      # 512
TEXT_DIM = T_train.shape[1]         # 512
classes_sorted = sorted(train_df["categories"].unique())
class_to_idx = {c: i for i, c in enumerate(classes_sorted)}
y_train_idx = train_df["categories"].map(class_to_idx).values
y_val_idx   = val_df["categories"].map(class_to_idx).values
y_test_idx  = test_df["categories"].map(class_to_idx).values
NUM_CLASSES = len(classes_sorted)

class TrimodalFusionClassifier(nn.Module):
    """(image_emb, metadata_fused_emb, text_emb) -> class logits.
    Reuses Phase 3's already-fused image+metadata vector (F) as one input,
    concatenates the raw text embedding, then applies the same linear/ReLU/dropout/linear pattern.
    """
    def __init__(self, fused_dim, text_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(fused_dim + text_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, fused_emb, text_emb):
        return self.net(torch.cat([fused_emb, text_emb], dim=1))

def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=DEVICE)

Ff_train, Ff_val, Ff_test = to_tensor(F_train), to_tensor(F_val), to_tensor(F_test)
Tt_train, Tt_val, Tt_test = to_tensor(T_train), to_tensor(T_val), to_tensor(T_test)
Yt_train = to_tensor(y_train_idx, dtype=torch.long)
Yt_val   = to_tensor(y_val_idx,   dtype=torch.long)

trimodal_model = TrimodalFusionClassifier(F_train.shape[1], TEXT_DIM, NUM_CLASSES).to(DEVICE)
class_counts = np.bincount(y_train_idx, minlength=NUM_CLASSES)
class_weights = to_tensor(1.0 / np.maximum(class_counts, 1))
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(trimodal_model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_f1, best_state = -1.0, None
for epoch in range(100):
    trimodal_model.train()
    optimizer.zero_grad()
    logits = trimodal_model(Ff_train, Tt_train)
    loss = criterion(logits, Yt_train)
    loss.backward(); optimizer.step()

    trimodal_model.eval()
    with torch.no_grad():
        val_preds = trimodal_model(Ff_val, Tt_val).argmax(dim=1).cpu().numpy()
        _, _, val_f1, _ = precision_recall_fscore_support(y_val_idx, val_preds, average=None, zero_division=0)
        val_macro_f1 = val_f1.mean()
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_state = {k: v.clone() for k, v in trimodal_model.state_dict().items()}
    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}  loss={loss.item():.4f}  val_macro_f1={val_macro_f1:.4f}")

trimodal_model.load_state_dict(best_state)
print(f"\nbest val macro-F1: {best_val_f1:.4f}")

epoch   0  loss=2.0840  val_macro_f1=0.6247
epoch  10  loss=0.8790  val_macro_f1=0.8239
epoch  20  loss=0.2441  val_macro_f1=0.9244
epoch  30  loss=0.0765  val_macro_f1=0.9531
epoch  40  loss=0.0268  val_macro_f1=1.0000
epoch  50  loss=0.0145  val_macro_f1=1.0000
epoch  60  loss=0.0081  val_macro_f1=1.0000
epoch  70  loss=0.0062  val_macro_f1=1.0000
epoch  80  loss=0.0048  val_macro_f1=1.0000
epoch  90  loss=0.0032  val_macro_f1=1.0000

best val macro-F1: 1.0000


## Test-set evaluation: trimodal vs Phase 1 (image-only) vs Phase 3 (image+metadata)

In [8]:
trimodal_model.eval()
with torch.no_grad():
    test_preds_idx = trimodal_model(Ff_test, Tt_test).argmax(dim=1).cpu().numpy()

acc = accuracy_score(y_test_idx, test_preds_idx)
_, _, f1_per_class, support = precision_recall_fscore_support(
    y_test_idx, test_preds_idx, labels=range(NUM_CLASSES), average=None, zero_division=0)
macro_f1 = f1_per_class.mean()

print(f"Trimodal (image+metadata+text) -- Test accuracy: {acc:.4f}   Macro-F1: {macro_f1:.4f}\n")
print("Reference -- Phase 1 (image-only): acc=0.3063  macro_f1=0.2469")
print("Reference -- Phase 3 (image+metadata): acc=0.3188  macro_f1=0.2174\n")

for c, f, s in zip(classes_sorted, f1_per_class, support):
    print(f"  {c:20s}  F1={f:.3f}  n={s}")

pd.DataFrame([
    {"representation": "image_only_phase1", "accuracy": 0.3063, "macro_f1": 0.2469},
    {"representation": "image_plus_metadata_phase3", "accuracy": 0.3188, "macro_f1": 0.2174},
    {"representation": "image_plus_metadata_plus_text", "accuracy": acc, "macro_f1": macro_f1},
]).to_csv("phase5_baseline_comparison.csv", index=False)

Trimodal (image+metadata+text) -- Test accuracy: 0.9938   Macro-F1: 0.9621

Reference -- Phase 1 (image-only): acc=0.3063  macro_f1=0.2469
Reference -- Phase 3 (image+metadata): acc=0.3188  macro_f1=0.2174

  A-ROP                 F1=1.000  n=25
  Laser-treated ROP     F1=1.000  n=54
  Stage 1 ROP           F1=1.000  n=37
  Stage 2 ROP           F1=1.000  n=52
  Stage 3 ROP           F1=0.987  n=38
  Stage 4 ROP           F1=0.960  n=12
  Stage 5 ROP           F1=0.750  n=5
  normal                F1=1.000  n=97


## Few-shot with trimodal representations (plan step 24)
Same nearest-prototype method and seeds as Phase 2/3, applied to the trimodal fusion layer's intermediate representation.

In [9]:
def build_prototypes(X, y, classes, k, rng):
    prototypes = []
    for c in classes:
        idx = np.where(y == c)[0]
        n_take = min(k, len(idx))
        chosen = rng.choice(idx, size=n_take, replace=False)
        prototypes.append(X[chosen].mean(axis=0))
    return np.stack(prototypes)

def nearest_prototype_predict(X, prototypes, classes, metric="cosine"):
    if metric == "cosine":
        X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        P_n = prototypes / (np.linalg.norm(prototypes, axis=1, keepdims=True) + 1e-8)
        nearest_idx = (X_n @ P_n.T).argmax(axis=1)
    else:
        dists = np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)
        nearest_idx = dists.argmin(axis=1)
    return np.array([classes[i] for i in nearest_idx])

@torch.no_grad()
def get_trimodal_repr(fused_t, text_t):
    x = torch.cat([fused_t, text_t], dim=1)
    return trimodal_model.net[0](x).cpu().numpy()  # after first Linear, before ReLU/dropout/final

trimodal_model.eval()
TR_train = get_trimodal_repr(Ff_train, Tt_train)
TR_test  = get_trimodal_repr(Ff_test,  Tt_test)

SHOTS = [1, 3, 5]
SEEDS = [0, 1, 2, 3, 4]
y_train_str = train_df["categories"].values
y_test_str  = test_df["categories"].values

trimodal_fewshot_results = []
for k in SHOTS:
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        prototypes = build_prototypes(TR_train, y_train_str, classes_sorted, k, rng)
        preds = nearest_prototype_predict(TR_test, prototypes, classes_sorted, metric="cosine")
        acc_k = accuracy_score(y_test_str, preds)
        _, _, f1_k, _ = precision_recall_fscore_support(
            y_test_str, preds, labels=classes_sorted, average=None, zero_division=0)
        trimodal_fewshot_results.append({"shot": k, "seed": seed, "accuracy": acc_k, "macro_f1": f1_k.mean()})

trimodal_fewshot_df = pd.DataFrame(trimodal_fewshot_results)
trimodal_summary = trimodal_fewshot_df.groupby("shot").agg(
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
).reset_index()
print(trimodal_summary.to_string(index=False))
trimodal_summary.to_csv("phase5_fewshot_trimodal_results.csv", index=False)

 shot  accuracy_mean  accuracy_std  macro_f1_mean  macro_f1_std
    1        0.97000      0.023028       0.877022      0.072394
    3        0.98125      0.005846       0.917129      0.026921
    5        0.98625      0.006092       0.929860      0.009856


## Open-set with trimodal representations (best method from Phase 4b: Euclidean distance)
Same unknown-class setup (Stage 4/5 ROP excluded from support), same threshold-selection-on-val logic as Phase 4/4b.

In [10]:
UNKNOWN_CLASSES = ["Stage 4 ROP", "Stage 5 ROP"]
known_classes = [c for c in classes_sorted if c not in UNKNOWN_CLASSES]

y_val_str = val_df["categories"].values
train_known_mask = np.isin(y_train_str, known_classes)
val_known_mask   = np.isin(y_val_str, known_classes)
test_known_mask  = np.isin(y_test_str, known_classes)

TR_val = get_trimodal_repr(Ff_val, Tt_val)
TR_train_known = TR_train[train_known_mask]
y_train_known = y_train_str[train_known_mask]

def select_threshold(val_dist, val_is_known, n_candidates=50):
    candidates = np.linspace(val_dist.min(), val_dist.max(), n_candidates)
    best_thresh, best_f1 = candidates[0], -1.0
    for t in candidates:
        f1 = f1_score(val_is_known, val_dist <= t, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t
    return best_thresh

def euclidean_dist_matrix(X, prototypes):
    return np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)

openset_results = []
for k in SHOTS:
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        prototypes = build_prototypes(TR_train_known, y_train_known, known_classes, k, rng)

        val_dm  = euclidean_dist_matrix(TR_val,  prototypes)
        test_dm = euclidean_dist_matrix(TR_test, prototypes)
        val_dist = val_dm.min(axis=1)
        test_dist = test_dm.min(axis=1)
        test_pred_idx = test_dm.argmin(axis=1)
        test_preds = np.array([known_classes[i] for i in test_pred_idx])

        threshold = select_threshold(val_dist, val_known_mask)
        test_accepted = test_dist <= threshold

        unknown_f1 = f1_score(~test_known_mask, ~test_accepted, zero_division=0)
        eval_mask = test_known_mask & test_accepted
        known_f1 = (f1_score(y_test_str[eval_mask], test_preds[eval_mask],
                              labels=known_classes, average="macro", zero_division=0)
                    if eval_mask.sum() > 0 else 0.0)
        try:
            auroc = roc_auc_score(~test_known_mask, test_dist)
        except ValueError:
            auroc = float("nan")

        openset_results.append({"shot": k, "seed": seed, "known_f1": known_f1,
                                 "unknown_f1": unknown_f1, "auroc": auroc})

openset_df = pd.DataFrame(openset_results)
openset_summary = openset_df.groupby("shot").agg(
    known_f1_mean=("known_f1", "mean"), known_f1_std=("known_f1", "std"),
    unknown_f1_mean=("unknown_f1", "mean"), unknown_f1_std=("unknown_f1", "std"),
    auroc_mean=("auroc", "mean"), auroc_std=("auroc", "std"),
).reset_index()
print(openset_summary.to_string(index=False))
openset_summary.to_csv("phase5_openset_trimodal.csv", index=False)

 shot  known_f1_mean  known_f1_std  unknown_f1_mean  unknown_f1_std  auroc_mean  auroc_std
    1            1.0           0.0         0.210671        0.198717    0.910231   0.025027
    3            1.0           0.0         0.489405        0.107932    0.956164   0.012125
    5            1.0           0.0         0.480261        0.111358    0.962881   0.008561


## Done — Phase 5 deliverable
Record: (1) full-train trimodal vs Phase 1/3 baselines, (2) few-shot trimodal vs Phase 2/3, (3) open-set trimodal vs Phase 4/4b.
**Caveat to state explicitly in the report:** text embeddings are per-class, not per-patient -- any gains partly reflect the fusion layer learning a class-anchor shortcut, not necessarily richer per-patient signal. This is worth flagging as a limitation, not hidden.
Do **not** proceed to Phase 6 (LoRA) until all three comparisons are recorded.